In [ ]:
%pip install statsmodels

: 

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


#### r^2 berechnen

rse = sum (errors^2)/sum((actual-mean(actual))^2)
r^2 = 1-rse


In [ ]:
train_data = pd.read_csv('../0_DataPreparation/data/train_data.csv')
train_data.corr(numeric_only=True)['Umsatz'].sort_values(ascending=False)

In [ ]:
## mehrere Variablen
### Problem: Kaggle erwartet beim hochladen der Predictions genau 1830 Zeilen, wenn wir Zeilen im Test Data Bereich rausschneiden gibt es Probleme

# Trainingsdaten laden

train_data = pd.read_csv('../0_DataPreparation/data/data_ohne_LAG/train_data.csv')

# Modell trainieren ohne Windgeschwindigkeit und Fastzeit
mod = smf.ols('Umsatz ~ Temperatur  + KielerWoche + NiederschlagTagessummeInMM + WarengruppeBread + WarengruppeRolls + WarengruppeCroissants + WarengruppeConfectionery + WarengruppeCake + WarengruppeSeasonalBread + WochentagFreitag + WochentagSamstag + WochentagSonntag + Januar + Februar + Maerz + April + Mai + Juni + Juli + August + September + Oktober + November + Dezember + Bewoelkung0von0Klar + Bewoelkung1von8FastWolkenlos + Bewoelkung8von8Bedeckt + FeiertagChristiHimmelfahrt + FeiertagErsterMai + FeiertagHeiligabend + FeiertagOstermontag + FeiertagPfingstmontag + FeiertagSilvester + FeiertagTagDerDeutschenEinheit',
               data=train_data).fit()
print(mod.summary())

# MAPE berechnen
from sklearn.metrics import mean_absolute_percentage_error

# Letzten 20% der Trainingsdaten als Validierungsset
val_data = train_data.tail(int(len(train_data) * 0.2))

# Vorhersagen auf Validierungsset
val_predictions = mod.predict(val_data)

# Nur Zeilen ohne fehlende Umsatz-Werte
mask = val_data['Umsatz'].notna()
mape = mean_absolute_percentage_error(val_data['Umsatz'][mask], val_predictions[mask])

print(f"Baseline MAPE: {mape * 100:.2f}%")

# Test-/Vorhersagedaten laden (die 1830 Zeilen ohne Umsatz)
test_data = pd.read_csv('../0_DataPreparation/data/test_data.csv')  

# Vorhersagen machen
test_data['Umsatz_Vorhersage'] = mod.predict(test_data)

# Nur ID und Umsatz_Vorhersage speichern
predictions_output = test_data[['id', 'Umsatz_Vorhersage']]
predictions_output.to_csv('../0_DataPreparation/data/predictions_preencoded_imputated.csv', index=False)


# Erste Vorhersagen anzeigen
print("\nErste 10 Vorhersagen:")
print(predictions_output.head(10))

#print("Warengruppe Dummy-Spalten:", warengruppe_cols)
#print("Wetter Dummy-Spalten:", wetter_cols)
#print("Bewoelkung Dummy-Spalten:", bewoelkung_cols)
